# AIC 2026 — OCR ingestion · PP-OCRv6 medium

Notebook nguồn chạy độc lập trên **Google Colab** hoặc **Kaggle**. Job đọc đúng
keyframe do `aic2026-keyframes-transnetv2` cung cấp, phát hiện từng vùng chữ bằng
PP-OCRv6 medium và ghi hai artifact cho mỗi video:

- `ocr.parquet`: đúng một dòng cho mỗi keyframe;
- `ocr_regions.parquet`: một dòng cho mỗi vùng chữ, có confidence, polygon và
  bounding box chuẩn hóa;
- `_OCR_SUCCESS.json`: marker provenance/hash được upload sau cùng trong cùng
  commit với hai Parquet.

PP-OCRv6 là OCR detector + recognizer, không phải model sinh văn bản. Vì vậy job
không có thinking, sampling, token limit hoặc retry hallucination. Mỗi GPU giữ
một pipeline riêng; Kaggle 2×T4 xử lý hai video đồng thời, Colab 1 GPU xử lý một
video. Recognition batch gom các crop chữ bên trong từng frame.

Quy trình an toàn:

1. Chạy toàn bộ notebook với `PREFLIGHT_ONLY=True`; xem 5 ảnh có box, không upload.
2. Đặt `PREFLIGHT_ONLY=False`, giữ `MAX_VIDEOS_PER_RUN=5` để chạy pilot.
3. Sau khi kiểm tra artifact, pin revision in ra ở cell cấu hình và đặt
   `MAX_VIDEOS_PER_RUN=None` để chạy toàn bộ.
4. Nếu runtime ngắt, chạy lại từ đầu. Marker hợp lệ được skip; checkpoint cục bộ
   tiếp tục video đang dang dở nếu storage runtime còn tồn tại.

In [ ]:
# PP-OCRv6 requires PaddleOCR 3.7 and PaddlePaddle 3.x. The CUDA 12.6 wheel also
# runs on newer NVIDIA drivers used by current Colab/Kaggle images.
import importlib.metadata
import subprocess
import sys


def installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None


required = {
    "paddleocr": "3.7.0",
    "paddlepaddle-gpu": "3.2.0",
}
needs_install = any(installed_version(name) != version for name, version in required.items())
if needs_install:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "paddleocr==3.7.0", "paddlepaddle-gpu==3.2.0",
        "huggingface_hub>=0.34,<2", "pandas>=2.2,<3", "pyarrow>=17,<22",
        "pillow>=10,<13", "numpy>=1.26,<3",
        "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu126/",
        "--extra-index-url", "https://pypi.org/simple",
    ])
print({name: installed_version(name) for name in required})

In [ ]:
import gc
import hashlib
import io
import json
import os
import re
import shutil
import sys
import tarfile
import tempfile
import time
import unicodedata
import warnings
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
from queue import Queue

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "180"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "45"
os.environ["FLAGS_allocator_strategy"] = "auto_growth"

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", message="No ccache found.*")
import paddle
from huggingface_hub import (
    CommitOperationAdd, HfApi, hf_hub_download, snapshot_download,
)
from paddleocr import PaddleOCR
from PIL import Image, ImageDraw, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = False

INPUT_REPO = "aqpahm/aic2026-keyframes-transnetv2"
OUTPUT_REPO = "aqpahm/aic2026-ocr-ppocrv6-medium"
DETECTION_MODEL_ID = "PaddlePaddle/PP-OCRv6_medium_det"
RECOGNITION_MODEL_ID = "PaddlePaddle/PP-OCRv6_medium_rec"
DETECTION_MODEL = "PP-OCRv6_medium_det"
RECOGNITION_MODEL = "PP-OCRv6_medium_rec"
DETECTION_MODEL_REVISION_PIN = None
RECOGNITION_MODEL_REVISION_PIN = None
INPUT_REVISION_PIN = None
MODEL_ID = f"{DETECTION_MODEL_ID}+{RECOGNITION_MODEL_ID}"
OUTPUT_PRIVATE = True

PREFLIGHT_ONLY = True
RUN_SMOKE_TEST_BEFORE_INGEST = True
PREFLIGHT_FRAMES = 5
PREFLIGHT_FRAME_SPECS = [
    ("L26_V221", 419),
    ("L30_V096", 637),
    ("L30_V096", 646),
]
MAX_VIDEOS_PER_RUN = 5
SELECT_LEVELS = None  # Example: {"L21"}; None means every supplied level.

TEXT_DET_LIMIT_SIDE_LEN = 1920
TEXT_DET_LIMIT_TYPE = "max"
TEXT_DET_THRESH = 0.30
TEXT_DET_BOX_THRESH = 0.55
TEXT_DET_UNCLIP_RATIO = 1.50
TEXT_REC_SCORE_THRESH = 0.45
TEXT_RECOGNITION_BATCH_SIZE = 32
USE_DOC_ORIENTATION_CLASSIFY = False
USE_DOC_UNWARPING = False
USE_TEXTLINE_ORIENTATION = False

PREFETCH_VIDEOS_PER_GPU = 1
UPLOAD_WORKERS = 1
MAX_PENDING_UPLOADS = 2
CHECKPOINT_FSYNC_EVERY_FRAMES = 25
MAX_CONSECUTIVE_IDENTICAL_FAILURES = 3
SCHEMA_VERSION = 6
NO_TEXT_SENTINEL = "<NO_TEXT>"  # Display only; stored OCR text is empty.


def runtime_name():
    if "google.colab" in sys.modules:
        return "colab"
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    return "local"


def secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        if RUNTIME == "colab":
            from google.colab import userdata
            return userdata.get(name)
        if RUNTIME == "kaggle":
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
    except Exception as error:
        raise RuntimeError(f"Cannot read {name}; grant this notebook access") from error
    raise RuntimeError(f"Missing {name}; add it to Colab/Kaggle Secrets")


RUNTIME = runtime_name()
ROOT = (
    Path("/content/aic-ocr-ppocrv6-medium") if RUNTIME == "colab"
    else Path("/kaggle/temp/aic-ocr-ppocrv6-medium") if RUNTIME == "kaggle"
    else Path(tempfile.gettempdir()) / "aic-ocr-ppocrv6-medium"
)
DOWNLOAD_ROOT = ROOT / "downloads"
OUTPUT_ROOT = ROOT / "output"
CHECKPOINT_ROOT = ROOT / "checkpoints"
for directory in (DOWNLOAD_ROOT, OUTPUT_ROOT, CHECKPOINT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

GPU_COUNT = paddle.device.cuda.device_count()
if GPU_COUNT < 1:
    raise RuntimeError("Enable a GPU runtime before running OCR")
GPU_IDS = list(range(GPU_COUNT))
for gpu_id in GPU_IDS:
    properties = paddle.device.cuda.get_device_properties(gpu_id)
    print(f"gpu:{gpu_id}: {properties.name}")
if PREFLIGHT_ONLY:
    GPU_IDS = GPU_IDS[:1]
    print("Preflight uses cuda:0 only; pilot/full uses every detected GPU.")

HF_TOKEN = secret("HF_TOKEN")
api = HfApi(token=HF_TOKEN)
print("Runtime:", RUNTIME, "| HF account:", api.whoami()["name"])
if not PREFLIGHT_ONLY:
    api.create_repo(
        repo_id=OUTPUT_REPO,
        repo_type="dataset",
        private=OUTPUT_PRIVATE,
        exist_ok=True,
    )
INPUT_REVISION = INPUT_REVISION_PIN or api.dataset_info(INPUT_REPO).sha
DETECTION_MODEL_REVISION = (
    DETECTION_MODEL_REVISION_PIN or api.model_info(DETECTION_MODEL_ID).sha
)
RECOGNITION_MODEL_REVISION = (
    RECOGNITION_MODEL_REVISION_PIN or api.model_info(RECOGNITION_MODEL_ID).sha
)
MODEL_REVISION = (
    f"det@{DETECTION_MODEL_REVISION};rec@{RECOGNITION_MODEL_REVISION}"
)
MODEL_ROOT = ROOT / "models"
DETECTION_MODEL_DIR = Path(snapshot_download(
    repo_id=DETECTION_MODEL_ID, revision=DETECTION_MODEL_REVISION,
    token=HF_TOKEN, local_dir=MODEL_ROOT / "det",
))
RECOGNITION_MODEL_DIR = Path(snapshot_download(
    repo_id=RECOGNITION_MODEL_ID, revision=RECOGNITION_MODEL_REVISION,
    token=HF_TOKEN, local_dir=MODEL_ROOT / "rec",
))
PADDLEOCR_VERSION = importlib.metadata.version("paddleocr")
PADDLE_VERSION = paddle.__version__
print("Input revision:", INPUT_REVISION)
print("Detection model revision:", DETECTION_MODEL_REVISION)
print("Recognition model revision:", RECOGNITION_MODEL_REVISION)
print("Output:", f"https://huggingface.co/datasets/{OUTPUT_REPO}")

RUN_CONFIG = {
    "source_revision": INPUT_REVISION,
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "detection_model_id": DETECTION_MODEL_ID,
    "detection_model": DETECTION_MODEL,
    "detection_model_revision": DETECTION_MODEL_REVISION,
    "recognition_model_id": RECOGNITION_MODEL_ID,
    "recognition_model": RECOGNITION_MODEL,
    "recognition_model_revision": RECOGNITION_MODEL_REVISION,
    "paddleocr_version": PADDLEOCR_VERSION,
    "paddle_version": PADDLE_VERSION,
    "text_det_limit_side_len": TEXT_DET_LIMIT_SIDE_LEN,
    "text_det_limit_type": TEXT_DET_LIMIT_TYPE,
    "text_det_thresh": TEXT_DET_THRESH,
    "text_det_box_thresh": TEXT_DET_BOX_THRESH,
    "text_det_unclip_ratio": TEXT_DET_UNCLIP_RATIO,
    "text_rec_score_thresh": TEXT_REC_SCORE_THRESH,
    "text_recognition_batch_size": TEXT_RECOGNITION_BATCH_SIZE,
    "use_doc_orientation_classify": USE_DOC_ORIENTATION_CLASSIFY,
    "use_doc_unwarping": USE_DOC_UNWARPING,
    "use_textline_orientation": USE_TEXTLINE_ORIENTATION,
    "schema_version": SCHEMA_VERSION,
}
RUN_CONFIG_SHA256 = hashlib.sha256(
    json.dumps(RUN_CONFIG, sort_keys=True).encode("utf-8")
).hexdigest()

## 1. Nạp một pipeline PP-OCRv6 trên mỗi GPU

Không bật document orientation/unwarping vì keyframe video đã đúng chiều. Detector
giữ cạnh dài tối đa 1920 để còn đọc được subtitle và bảng hiệu nhỏ. Batch 32 là
batch các crop chữ của recognizer, không phải 32 ảnh nguyên khung.

In [ ]:
def build_pipeline(gpu_id):
    return PaddleOCR(
        device=f"gpu:{gpu_id}",
        engine="paddle",
        text_detection_model_name=DETECTION_MODEL,
        text_detection_model_dir=str(DETECTION_MODEL_DIR),
        text_recognition_model_name=RECOGNITION_MODEL,
        text_recognition_model_dir=str(RECOGNITION_MODEL_DIR),
        text_recognition_batch_size=TEXT_RECOGNITION_BATCH_SIZE,
        use_doc_orientation_classify=USE_DOC_ORIENTATION_CLASSIFY,
        use_doc_unwarping=USE_DOC_UNWARPING,
        use_textline_orientation=USE_TEXTLINE_ORIENTATION,
        text_det_limit_side_len=TEXT_DET_LIMIT_SIDE_LEN,
        text_det_limit_type=TEXT_DET_LIMIT_TYPE,
        text_det_thresh=TEXT_DET_THRESH,
        text_det_box_thresh=TEXT_DET_BOX_THRESH,
        text_det_unclip_ratio=TEXT_DET_UNCLIP_RATIO,
        text_rec_score_thresh=TEXT_REC_SCORE_THRESH,
    )


pipelines = []
for gpu_id in GPU_IDS:
    print(f"Loading PP-OCRv6 medium on cuda:{gpu_id} ...")
    pipelines.append(build_pipeline(gpu_id))
print("Loaded pipelines:", len(pipelines))


def result_payload(result):
    value = result.json if hasattr(result, "json") else result
    if callable(value):
        value = value()
    if not isinstance(value, dict):
        raise RuntimeError(f"Unexpected PaddleOCR result type: {type(value)!r}")
    payload = value.get("res", value)
    if not isinstance(payload, dict):
        raise RuntimeError("PaddleOCR result does not contain a result mapping")
    return payload


def as_list(value):
    if value is None:
        return []
    if isinstance(value, np.ndarray):
        return value.tolist()
    return list(value)


def polygon_to_region(text, score, polygon, width, height):
    points = np.asarray(polygon, dtype=np.float64).reshape(-1, 2)
    if len(points) < 4 or not np.isfinite(points).all():
        raise RuntimeError("PaddleOCR returned an invalid polygon")
    xs = np.clip(points[:, 0], 0, width)
    ys = np.clip(points[:, 1], 0, height)
    x1, x2 = float(xs.min() / width), float(xs.max() / width)
    y1, y2 = float(ys.min() / height), float(ys.max() / height)
    if x2 <= x1 or y2 <= y1:
        raise RuntimeError("PaddleOCR returned a zero-area polygon")
    normalized_polygon = [
        [round(float(x / width), 8), round(float(y / height), 8)]
        for x, y in zip(xs, ys)
    ]
    return {
        "text": unicodedata.normalize("NFC", str(text)).strip(),
        "confidence": float(score),
        "polygon_json": json.dumps(normalized_polygon, separators=(",", ":")),
        "bbox_x1": x1,
        "bbox_y1": y1,
        "bbox_x2": x2,
        "bbox_y2": y2,
    }


def reading_order_key(region):
    height = max(region["bbox_y2"] - region["bbox_y1"], 1e-6)
    y_center = (region["bbox_y1"] + region["bbox_y2"]) / 2
    return (round(y_center / 0.02), region["bbox_x1"], y_center)


def run_ocr(image, pipeline):
    width, height = image.size
    # PaddleOCR's ndarray interface expects the OpenCV BGR channel order.
    bgr = np.ascontiguousarray(np.asarray(image, dtype=np.uint8)[:, :, ::-1])
    outputs = list(pipeline.predict(input=bgr))
    if len(outputs) != 1:
        raise RuntimeError(f"Expected one OCR result, received {len(outputs)}")
    payload = result_payload(outputs[0])
    texts = as_list(payload.get("rec_texts"))
    scores = as_list(payload.get("rec_scores"))
    polygons = as_list(payload.get("rec_polys"))
    if not polygons and payload.get("rec_boxes") is not None:
        polygons = [
            [[box[0], box[1]], [box[2], box[1]], [box[2], box[3]], [box[0], box[3]]]
            for box in payload["rec_boxes"]
        ]
    if not (len(texts) == len(scores) == len(polygons)):
        raise RuntimeError(
            "PaddleOCR returned mismatched rec_texts/rec_scores/rec_polys lengths"
        )
    regions = []
    for text, score, polygon in zip(texts, scores, polygons):
        text = str(text).strip()
        score = float(score)
        if not text or score < TEXT_REC_SCORE_THRESH:
            continue
        regions.append(polygon_to_region(text, score, polygon, width, height))
    regions.sort(key=reading_order_key)
    for index, region in enumerate(regions):
        region["region_id"] = index
        region["reading_order"] = index
    text = "\n".join(region["text"] for region in regions)
    scores = [region["confidence"] for region in regions]
    return {
        "ocr_text": text,
        "ocr_status": "ok" if regions else "no_text",
        "regions": regions,
        "image_width": width,
        "image_height": height,
        "mean_confidence": float(np.mean(scores)) if scores else None,
        "min_confidence": float(np.min(scores)) if scores else None,
    }

## 2. Đọc keyframe và checkpoint

Checkpoint có fingerprint của input, model, tham số và schema. Chỉ các dòng liên
tục đúng thứ tự frame mới được resume. Lỗi inference không được biến thành kết
quả rỗng; video sẽ fail và không có marker thành công.

In [ ]:
def retry(operation, description, attempts=6):
    for attempt in range(1, attempts + 1):
        try:
            return operation()
        except Exception as error:
            if attempt == attempts:
                raise
            delay = min(180, 5 * 2 ** (attempt - 1))
            print(f"{description}: {type(error).__name__}; retry in {delay}s [{attempt}/{attempts}]")
            time.sleep(delay)


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while block := stream.read(8 * 1024 * 1024):
            digest.update(block)
    return digest.hexdigest()


def download_input(filename, local_dir):
    local_dir.mkdir(parents=True, exist_ok=True)
    return Path(retry(
        lambda: hf_hub_download(
            repo_id=INPUT_REPO,
            repo_type="dataset",
            filename=filename,
            revision=INPUT_REVISION,
            token=HF_TOKEN,
            local_dir=local_dir,
        ),
        f"download {filename}",
    ))


def normalized_tar_name(value):
    return PurePosixPath(str(value).replace("\\", "/")).as_posix().lstrip("./")


def tar_lookup(archive):
    by_path = {}
    by_name = defaultdict(list)
    for member in archive.getmembers():
        if member.isfile():
            name = normalized_tar_name(member.name)
            by_path[name] = member
            by_name[PurePosixPath(name).name].append(member)
    return by_path, by_name


def read_frame(archive, image_path, by_path, by_name):
    expected = normalized_tar_name(image_path)
    member = by_path.get(expected)
    if member is None:
        matches = by_name.get(PurePosixPath(expected).name, [])
        if len(matches) != 1:
            raise RuntimeError(f"Cannot uniquely find {image_path!r} in keyframes.tar")
        member = matches[0]
    stream = archive.extractfile(member)
    if stream is None:
        raise RuntimeError(f"Cannot read {member.name}")
    with Image.open(io.BytesIO(stream.read())) as source:
        image = source.convert("RGB")
        image.load()
    return image


ID_COLUMNS = [
    "video_id", "frame_uid", "sample_n", "frame_idx", "shot_id",
    "timestamp_sec", "image_path",
]
OCR_COLUMNS = ID_COLUMNS + [
    "ocr_text", "ocr_text_normalized", "ocr_text_no_accent", "has_text",
    "ocr_status", "region_count", "image_width", "image_height",
    "mean_confidence", "min_confidence", "model_id", "model_revision",
]
REGION_COLUMNS = ID_COLUMNS + [
    "region_id", "reading_order", "text", "text_normalized", "text_no_accent",
    "confidence", "polygon_json", "bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2",
    "model_id", "model_revision",
]


def make_manifest(source_frame, video_id):
    required = {"video_id", "sample_n", "frame_idx", "shot_id", "timestamp_sec", "image_path"}
    missing = required - set(source_frame.columns)
    if missing or source_frame.empty:
        raise RuntimeError(f"{video_id}: empty/incomplete frames.parquet; missing {sorted(missing)}")
    frame = source_frame.sort_values("sample_n", kind="stable").reset_index(drop=True).copy()
    if frame["video_id"].astype(str).nunique() != 1 or str(frame.iloc[0]["video_id"]) != video_id:
        raise RuntimeError(f"{video_id}: source video_id mismatch")
    if frame["sample_n"].duplicated().any() or frame["frame_idx"].duplicated().any():
        raise RuntimeError(f"{video_id}: duplicate sample_n/frame_idx")
    frame["video_id"] = frame["video_id"].astype(str)
    frame["sample_n"] = frame["sample_n"].astype("int64")
    frame["frame_idx"] = frame["frame_idx"].astype("int64")
    frame["shot_id"] = frame["shot_id"].astype("int64")
    frame["timestamp_sec"] = frame["timestamp_sec"].astype("float64")
    frame["image_path"] = frame["image_path"].astype(str)
    frame["frame_uid"] = [f"{video_id}:{index}" for index in frame["frame_idx"]]
    return frame[ID_COLUMNS]


def normalize_search(text):
    return " ".join(unicodedata.normalize("NFC", text).casefold().split())


def remove_accents(text):
    value = unicodedata.normalize("NFD", text)
    value = "".join(char for char in value if unicodedata.category(char) != "Mn")
    return value.replace("đ", "d").replace("Đ", "D")


def checkpoint_path(video_id):
    return CHECKPOINT_ROOT / f"{video_id}.jsonl"


def checkpoint_header(video_id, manifest):
    return {
        "record_type": "header",
        "video_id": video_id,
        "run_config_sha256": RUN_CONFIG_SHA256,
        "frame_count": len(manifest),
        "first_frame_uid": manifest.iloc[0]["frame_uid"],
        "last_frame_uid": manifest.iloc[-1]["frame_uid"],
    }


def rewrite_checkpoint(path, header, records):
    temporary = path.with_suffix(".tmp")
    with temporary.open("w", encoding="utf-8") as stream:
        stream.write(json.dumps(header, ensure_ascii=False) + "\n")
        for record in records:
            stream.write(json.dumps(record, ensure_ascii=False) + "\n")
        stream.flush()
        os.fsync(stream.fileno())
    temporary.replace(path)


def load_checkpoint(video_id, manifest):
    path = checkpoint_path(video_id)
    expected_header = checkpoint_header(video_id, manifest)
    if not path.exists():
        rewrite_checkpoint(path, expected_header, [])
        return []
    valid = []
    try:
        with path.open(encoding="utf-8") as stream:
            if json.loads(stream.readline()) != expected_header:
                raise ValueError("checkpoint header does not match this run")
            for position, line in enumerate(stream):
                if not line.strip():
                    continue
                try:
                    record = json.loads(line)
                except json.JSONDecodeError:
                    print(f"  Truncating incomplete checkpoint row {position} for {video_id}")
                    break
                if position >= len(manifest):
                    break
                if record.get("frame_uid") != manifest.iloc[position]["frame_uid"]:
                    raise ValueError(f"checkpoint order mismatch at row {position}")
                result = record.get("result")
                if not isinstance(result, dict) or not {
                    "ocr_text", "ocr_status", "regions", "image_width", "image_height"
                } <= set(result):
                    raise ValueError(f"invalid checkpoint result at row {position}")
                valid.append(record)
    except (OSError, UnicodeError, json.JSONDecodeError, ValueError) as error:
        print(f"  Ignoring incompatible checkpoint for {video_id}: {error}")
        valid = []
    rewrite_checkpoint(path, expected_header, valid)
    if valid:
        print(f"  Resuming {video_id} at frame {len(valid)}/{len(manifest)}")
    return [record["result"] for record in valid]


def infer_video(archive, manifest, video_id, pipeline):
    by_path, by_name = tar_lookup(archive)
    results = load_checkpoint(video_id, manifest)
    resumed_frames = len(results)
    inference_sec = 0.0
    checkpoint = checkpoint_path(video_id)
    stream = checkpoint.open("a", encoding="utf-8")
    last_sync = len(results)
    try:
        for position in range(len(results), len(manifest)):
            row = manifest.iloc[position]
            image = read_frame(archive, row["image_path"], by_path, by_name)
            started = time.perf_counter()
            try:
                result = run_ocr(image, pipeline)
            finally:
                image.close()
            inference_sec += time.perf_counter() - started
            results.append(result)
            stream.write(json.dumps({
                "frame_uid": row["frame_uid"],
                "result": result,
            }, ensure_ascii=False) + "\n")
            cursor = position + 1
            if cursor == len(manifest) or cursor - last_sync >= CHECKPOINT_FSYNC_EVERY_FRAMES:
                stream.flush()
                os.fsync(stream.fileno())
                last_sync = cursor
            if cursor % 100 == 0 or cursor == len(manifest):
                print(f"  OCR {cursor}/{len(manifest)}")
    finally:
        stream.flush()
        os.fsync(stream.fileno())
        stream.close()
    return results, inference_sec, resumed_frames


def draw_regions(image, regions):
    preview = image.copy()
    draw = ImageDraw.Draw(preview)
    width, height = preview.size
    line_width = max(2, round(max(width, height) / 500))
    for region in regions:
        polygon = json.loads(region["polygon_json"])
        points = [(round(x * width), round(y * height)) for x, y in polygon]
        draw.line(points + [points[0]], fill="red", width=line_width)
    return preview


def spread_indices(indices, count):
    take = min(len(indices), max(0, count))
    if take <= 1:
        return indices[:take]
    return [indices[round(i * (len(indices) - 1) / (take - 1))] for i in range(take)]

## 3. Chọn đúng video nguồn và kiểm tra resume

Danh sách video được lấy từ thành viên thực tế của dataset keyframe rồi sắp xếp
theo `level/video_id`; không suy đoán ID liên tục. Chỉ marker có đúng provenance,
cấu hình, schema và hash của cả hai Parquet mới được skip.

In [ ]:
input_files = set(retry(
    lambda: api.list_repo_files(INPUT_REPO, repo_type="dataset", revision=INPUT_REVISION),
    "list keyframe files",
))
output_files = set() if PREFLIGHT_ONLY else set(retry(
    lambda: api.list_repo_files(OUTPUT_REPO, repo_type="dataset"),
    "list OCR files",
))
pattern = re.compile(r"^data/(L(?:2[1-9]|30))/(L\d+_V\d+)/keyframes\.tar$")
video_specs = []
for filename in sorted(input_files):
    match = pattern.fullmatch(filename)
    if not match:
        continue
    level, video_id = match.groups()
    if SELECT_LEVELS is not None and level not in SELECT_LEVELS:
        continue
    prefix = f"data/{level}/{video_id}"
    frames_filename = f"{prefix}/frames.parquet"
    source_marker = f"{prefix}/_SUCCESS.json"
    if frames_filename not in input_files or source_marker not in input_files:
        raise RuntimeError(f"Incomplete keyframe source for {video_id}")
    video_specs.append({
        "level": level,
        "video_id": video_id,
        "tar_filename": filename,
        "frames_filename": frames_filename,
        "source_marker": source_marker,
    })
if not video_specs:
    raise RuntimeError("No complete keyframe videos found")
spec_by_id = {spec["video_id"]: spec for spec in video_specs}


def remote_complete(spec, files):
    prefix = f"data/{spec['level']}/{spec['video_id']}"
    parquet_name = f"{prefix}/ocr.parquet"
    regions_name = f"{prefix}/ocr_regions.parquet"
    marker_name = f"{prefix}/_OCR_SUCCESS.json"
    if not {parquet_name, regions_name, marker_name} <= files:
        return False
    try:
        marker_path = retry(
            lambda: hf_hub_download(
                repo_id=OUTPUT_REPO,
                repo_type="dataset",
                filename=marker_name,
                token=HF_TOKEN,
            ),
            f"download marker {spec['video_id']}",
        )
        marker = json.loads(Path(marker_path).read_text(encoding="utf-8"))
        required = (
            marker.get("status") == "complete"
            and marker.get("video_id") == spec["video_id"]
            and marker.get("source_repo") == INPUT_REPO
            and marker.get("source_revision") == INPUT_REVISION
            and marker.get("model_id") == MODEL_ID
            and marker.get("model_revision") == MODEL_REVISION
            and marker.get("run_config_sha256") == RUN_CONFIG_SHA256
            and int(marker.get("schema_version", -1)) == SCHEMA_VERSION
        )
        if not required:
            return False
        verify_dir = DOWNLOAD_ROOT / "marker-validation" / spec["video_id"]
        parquet_path = Path(hf_hub_download(
            repo_id=OUTPUT_REPO,
            repo_type="dataset",
            filename=parquet_name,
            token=HF_TOKEN,
            local_dir=verify_dir,
        ))
        regions_path = Path(hf_hub_download(
            repo_id=OUTPUT_REPO,
            repo_type="dataset",
            filename=regions_name,
            token=HF_TOKEN,
            local_dir=verify_dir,
        ))
        return (
            sha256_file(parquet_path) == marker.get("parquet_sha256")
            and sha256_file(regions_path) == marker.get("regions_parquet_sha256")
        )
    except Exception as error:
        print(f"Invalid remote OCR output for {spec['video_id']}: {error}")
        return False
    finally:
        shutil.rmtree(DOWNLOAD_ROOT / "marker-validation" / spec["video_id"], ignore_errors=True)


if PREFLIGHT_ONLY:
    complete_flags = [False] * len(video_specs)
else:
    with ThreadPoolExecutor(max_workers=12) as validator:
        complete_flags = list(validator.map(
            lambda spec: remote_complete(spec, output_files), video_specs
        ))
missing_specs = [spec for spec, complete in zip(video_specs, complete_flags) if not complete]
selected_specs = (
    missing_specs if MAX_VIDEOS_PER_RUN is None
    else missing_specs[:MAX_VIDEOS_PER_RUN]
)
pending_specs = [] if PREFLIGHT_ONLY else selected_specs
print("Source videos:", len(video_specs))
print("Remote complete:", sum(complete_flags))
print("Selected this run:", len(pending_specs))
if pending_specs:
    print("First/last selected:", pending_specs[0]["video_id"], pending_specs[-1]["video_id"])

## 4. Ghi artifact, preflight và upload

Một commit chứa đủ hai Parquet và marker của đúng một video. Marker chỉ được tạo
sau khi hai artifact đã được ghi và kiểm tra tại local; marker nằm cuối danh sách
operation. Upload chạy trên một thread riêng trong lúc GPU tiếp tục video kế tiếp.

In [ ]:
def build_artifact_frames(manifest, results):
    if len(manifest) != len(results):
        raise RuntimeError("OCR result count does not match keyframe count")
    ocr_rows, region_rows = [], []
    for identity, result in zip(manifest.to_dict("records"), results):
        regions = result["regions"]
        text = result["ocr_text"]
        ocr_rows.append({
            **identity,
            "ocr_text": text,
            "ocr_text_normalized": normalize_search(text),
            "ocr_text_no_accent": remove_accents(normalize_search(text)),
            "has_text": bool(regions),
            "ocr_status": result["ocr_status"],
            "region_count": len(regions),
            "image_width": result["image_width"],
            "image_height": result["image_height"],
            "mean_confidence": result["mean_confidence"],
            "min_confidence": result["min_confidence"],
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        })
        for region in regions:
            region_rows.append({
                **identity,
                "region_id": region["region_id"],
                "reading_order": region["reading_order"],
                "text": region["text"],
                "text_normalized": normalize_search(region["text"]),
                "text_no_accent": remove_accents(normalize_search(region["text"])),
                "confidence": region["confidence"],
                "polygon_json": region["polygon_json"],
                "bbox_x1": region["bbox_x1"],
                "bbox_y1": region["bbox_y1"],
                "bbox_x2": region["bbox_x2"],
                "bbox_y2": region["bbox_y2"],
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            })
    ocr_frame = pd.DataFrame(ocr_rows, columns=OCR_COLUMNS)
    region_frame = pd.DataFrame(region_rows, columns=REGION_COLUMNS)
    for column in ["sample_n", "frame_idx", "shot_id", "region_count", "image_width", "image_height"]:
        ocr_frame[column] = ocr_frame[column].astype("int64")
    for column in ["timestamp_sec", "mean_confidence", "min_confidence"]:
        ocr_frame[column] = ocr_frame[column].astype("float64")
    ocr_frame["has_text"] = ocr_frame["has_text"].astype("bool")
    for column in ["video_id", "frame_uid", "image_path", "ocr_text", "ocr_text_normalized", "ocr_text_no_accent", "ocr_status", "model_id", "model_revision"]:
        ocr_frame[column] = ocr_frame[column].astype("string")
    for column in ["sample_n", "frame_idx", "shot_id", "region_id", "reading_order"]:
        region_frame[column] = region_frame[column].astype("int64")
    for column in ["timestamp_sec", "confidence", "bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2"]:
        region_frame[column] = region_frame[column].astype("float64")
    for column in ["video_id", "frame_uid", "image_path", "text", "text_normalized", "text_no_accent", "polygon_json", "model_id", "model_revision"]:
        region_frame[column] = region_frame[column].astype("string")
    return ocr_frame, region_frame


def validate_artifacts(manifest, ocr_frame, region_frame):
    if list(ocr_frame["frame_uid"]) != list(manifest["frame_uid"]):
        raise RuntimeError("ocr.parquet identity/order mismatch")
    if not set(ocr_frame["ocr_status"]).issubset({"ok", "no_text"}):
        raise RuntimeError("Unexpected OCR status")
    actual_counts = region_frame.groupby("frame_uid").size().to_dict()
    expected_counts = dict(zip(ocr_frame["frame_uid"], ocr_frame["region_count"]))
    if any(actual_counts.get(uid, 0) != count for uid, count in expected_counts.items()):
        raise RuntimeError("region_count does not match ocr_regions.parquet")
    if (ocr_frame["has_text"] != ocr_frame["region_count"].gt(0)).any():
        raise RuntimeError("has_text does not match region_count")
    if not region_frame.empty:
        boxes = region_frame[["bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2"]]
        if not ((boxes >= 0).all().all() and (boxes <= 1).all().all()):
            raise RuntimeError("Normalized OCR box is outside [0, 1]")
        if not (
            (region_frame["bbox_x2"] > region_frame["bbox_x1"]).all()
            and (region_frame["bbox_y2"] > region_frame["bbox_y1"]).all()
        ):
            raise RuntimeError("OCR region has non-positive area")
        if not region_frame["confidence"].between(TEXT_REC_SCORE_THRESH, 1.0).all():
            raise RuntimeError("OCR confidence is outside the valid range")


def write_artifacts(spec, manifest, results, stats):
    ocr_frame, region_frame = build_artifact_frames(manifest, results)
    validate_artifacts(manifest, ocr_frame, region_frame)
    directory = OUTPUT_ROOT / spec["level"] / spec["video_id"]
    directory.mkdir(parents=True, exist_ok=True)
    parquet_path = directory / "ocr.parquet"
    regions_path = directory / "ocr_regions.parquet"
    marker_path = directory / "_OCR_SUCCESS.json"
    ocr_frame.to_parquet(parquet_path, index=False, compression="zstd")
    region_frame.to_parquet(regions_path, index=False, compression="zstd")
    marker = {
        "status": "complete",
        "video_id": spec["video_id"],
        "source_repo": INPUT_REPO,
        "source_revision": INPUT_REVISION,
        "source_frames_path": spec["frames_filename"],
        "source_keyframes_path": spec["tar_filename"],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "detection_model_id": DETECTION_MODEL_ID,
        "detection_model": DETECTION_MODEL,
        "detection_model_revision": DETECTION_MODEL_REVISION,
        "recognition_model_id": RECOGNITION_MODEL_ID,
        "recognition_model": RECOGNITION_MODEL,
        "recognition_model_revision": RECOGNITION_MODEL_REVISION,
        "paddleocr_version": PADDLEOCR_VERSION,
        "paddle_version": PADDLE_VERSION,
        "run_config": RUN_CONFIG,
        "run_config_sha256": RUN_CONFIG_SHA256,
        "frame_count": len(ocr_frame),
        "frames_with_text": int(ocr_frame["has_text"].sum()),
        "region_count": len(region_frame),
        "parquet_sha256": sha256_file(parquet_path),
        "regions_parquet_sha256": sha256_file(regions_path),
        "stats": stats,
        "schema_version": SCHEMA_VERSION,
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    marker_path.write_text(json.dumps(marker, ensure_ascii=False, indent=2), encoding="utf-8")
    return [parquet_path, regions_path, marker_path]


device_pool = Queue()
for index in range(len(pipelines)):
    device_pool.put(index)


def process_video(spec):
    video_id = spec["video_id"]
    local_dir = DOWNLOAD_ROOT / video_id
    shutil.rmtree(local_dir, ignore_errors=True)
    local_dir.mkdir(parents=True, exist_ok=True)
    pipeline_index = None
    started = time.perf_counter()
    try:
        frames_path = download_input(spec["frames_filename"], local_dir)
        marker_path = download_input(spec["source_marker"], local_dir)
        source_marker = json.loads(marker_path.read_text(encoding="utf-8"))
        manifest = make_manifest(pd.read_parquet(frames_path), video_id)
        if source_marker.get("video_id") != video_id or source_marker.get("keyframe_count") != len(manifest):
            raise RuntimeError(f"{video_id}: invalid keyframe source marker")
        tar_path = download_input(spec["tar_filename"], local_dir)
        pipeline_index = device_pool.get()
        gpu_id = GPU_IDS[pipeline_index]
        with tarfile.open(tar_path, mode="r:*") as archive:
            results, inference_sec, resumed_frames = infer_video(
                archive, manifest, video_id, pipelines[pipeline_index]
            )
            qa_samples = []
            if MAX_VIDEOS_PER_RUN is not None:
                by_path, by_name = tar_lookup(archive)
                text_positions = [i for i, item in enumerate(results) if item["regions"]]
                empty_positions = [i for i, item in enumerate(results) if not item["regions"]]
                positions = list(dict.fromkeys(
                    spread_indices(text_positions, 6) + spread_indices(empty_positions, 2)
                ))[:8]
                for position in positions:
                    row = manifest.iloc[position]
                    image = read_frame(archive, row["image_path"], by_path, by_name)
                    try:
                        preview = draw_regions(image, results[position]["regions"])
                        preview.thumbnail((900, 900), Image.Resampling.LANCZOS)
                        buffer = io.BytesIO()
                        preview.save(buffer, format="JPEG", quality=85)
                        preview.close()
                        qa_samples.append({
                            "frame_uid": row["frame_uid"],
                            "timestamp_sec": float(row["timestamp_sec"]),
                            "ocr_text": results[position]["ocr_text"],
                            "region_count": len(results[position]["regions"]),
                            "jpeg": buffer.getvalue(),
                        })
                    finally:
                        image.close()
        elapsed = time.perf_counter() - started
        inferred_frames = len(results) - resumed_frames
        stats = {
            "elapsed_sec": round(elapsed, 3),
            "inference_sec": round(inference_sec, 3),
            "inference_frames": inferred_frames,
            "resumed_frames": resumed_frames,
            "inference_fps": round(inferred_frames / inference_sec, 3) if inference_sec else None,
            "device": f"gpu:{gpu_id}",
            "gpu": paddle.device.cuda.get_device_name(gpu_id),
            "text_recognition_batch_size": TEXT_RECOGNITION_BATCH_SIZE,
        }
        device_pool.put(pipeline_index)
        pipeline_index = None
        paths = write_artifacts(spec, manifest, results, stats)
        return {
            "video_id": video_id,
            "paths": paths,
            "frames": len(manifest),
            "regions": sum(len(item["regions"]) for item in results),
            "frames_with_text": sum(bool(item["regions"]) for item in results),
            "stats": stats,
            "qa_samples": qa_samples,
        }
    finally:
        if pipeline_index is not None:
            device_pool.put(pipeline_index)
        shutil.rmtree(local_dir, ignore_errors=True)
        gc.collect()


def upload_outputs(paths, video_id):
    operations = [
        CommitOperationAdd(
            path_in_repo=f"data/{path.relative_to(OUTPUT_ROOT).as_posix()}",
            path_or_fileobj=str(path),
        )
        for path in paths
    ]
    retry(
        lambda: api.create_commit(
            repo_id=OUTPUT_REPO,
            repo_type="dataset",
            operations=operations,
            commit_message=f"Add PP-OCRv6 medium OCR for {video_id}",
        ),
        f"upload {video_id}",
    )


def run_preflight():
    if PREFLIGHT_FRAMES < len(PREFLIGHT_FRAME_SPECS):
        raise ValueError("PREFLIGHT_FRAMES is smaller than required frame specs")
    grouped = defaultdict(list)
    for video_id, frame_idx in PREFLIGHT_FRAME_SPECS:
        grouped[video_id].append(frame_idx)
    selected, opened = [], {}
    try:
        for video_id, frame_indices in grouped.items():
            spec = spec_by_id.get(video_id)
            if spec is None:
                raise RuntimeError(f"Preflight video not found: {video_id}")
            local_dir = DOWNLOAD_ROOT / f"preflight-{video_id}"
            shutil.rmtree(local_dir, ignore_errors=True)
            frames_path = download_input(spec["frames_filename"], local_dir)
            tar_path = download_input(spec["tar_filename"], local_dir)
            manifest = make_manifest(pd.read_parquet(frames_path), video_id)
            positions = []
            for frame_idx in frame_indices:
                matches = manifest.index[manifest["frame_idx"] == frame_idx].tolist()
                if not matches:
                    raise RuntimeError(f"Preflight frame not found: {video_id}:{frame_idx}")
                positions.append(matches[0])
            opened[video_id] = (local_dir, tar_path, manifest, positions)
            selected.extend((video_id, position) for position in positions)
        remaining_count = PREFLIGHT_FRAMES - len(selected)
        first_video = next(iter(opened))
        _, _, first_manifest, used = opened[first_video]
        remaining = [i for i in range(len(first_manifest)) if i not in used]
        selected.extend((first_video, i) for i in spread_indices(remaining, remaining_count))

        samples = []
        for video_id, position in selected:
            _, tar_path, manifest, _ = opened[video_id]
            with tarfile.open(tar_path, mode="r:*") as archive:
                by_path, by_name = tar_lookup(archive)
                row = manifest.iloc[position]
                image = read_frame(archive, row["image_path"], by_path, by_name)
                try:
                    started = time.perf_counter()
                    result = run_ocr(image, pipelines[0])
                    elapsed = time.perf_counter() - started
                    preview = draw_regions(image, result["regions"])
                    preview.thumbnail((900, 900), Image.Resampling.LANCZOS)
                    samples.append({
                        "frame_uid": row["frame_uid"],
                        "timestamp_sec": float(row["timestamp_sec"]),
                        "ocr_text": result["ocr_text"],
                        "ocr_status": result["ocr_status"],
                        "region_count": len(result["regions"]),
                        "elapsed_sec": elapsed,
                        "preview": preview,
                    })
                finally:
                    image.close()
    finally:
        for local_dir, _, _, _ in opened.values():
            shutil.rmtree(local_dir, ignore_errors=True)

    from IPython.display import display
    print(f"Preflight: {len(samples)} frames; nothing will be uploaded")
    for sample in samples:
        print(
            "\n", sample["frame_uid"], f"t={sample['timestamp_sec']:.2f}s",
            f"elapsed={sample['elapsed_sec']:.3f}s", "status=", sample["ocr_status"],
            "regions=", sample["region_count"],
            "OCR=", sample["ocr_text"] or NO_TEXT_SENTINEL,
        )
        display(sample["preview"])
        sample["preview"].close()
    return [{key: value for key, value in sample.items() if key != "preview"} for sample in samples]


preflight_results = run_preflight() if PREFLIGHT_ONLY or RUN_SMOKE_TEST_BEFORE_INGEST else []
if not PREFLIGHT_ONLY and RUN_SMOKE_TEST_BEFORE_INGEST:
    print("Smoke test passed; starting queued video ingestion.")
if PREFLIGHT_ONLY:
    pending_specs = []
    print("Preflight finished. Review text and every red box, then set PREFLIGHT_ONLY=False.")

run_results, failures = [], []
uploaded_video_ids = set()


def finalize_upload(result):
    video_id = result["video_id"]
    upload_outputs(result["paths"], video_id)
    shutil.rmtree(OUTPUT_ROOT / video_id.split("_")[0] / video_id, ignore_errors=True)
    checkpoint_path(video_id).unlink(missing_ok=True)
    return video_id


run_started = time.perf_counter()
worker_count = max(1, len(pipelines) * (1 + PREFETCH_VIDEOS_PER_GPU))
source_queue = iter(pending_specs)


def collect_upload(upload_futures, future):
    result = upload_futures.pop(future)
    try:
        video_id = future.result()
    except Exception as error:
        failures.append({"video_id": result["video_id"], "stage": "upload", "error": repr(error)})
        print(f"UPLOAD FAILED {result['video_id']}: {error!r}")
    else:
        uploaded_video_ids.add(video_id)
        print(f"Uploaded {video_id}; local artifact and checkpoint removed")


with ThreadPoolExecutor(max_workers=UPLOAD_WORKERS) as upload_executor:
    upload_futures = {}
    with ThreadPoolExecutor(max_workers=worker_count) as executor:
        futures = {}
        halt_scheduling = False
        last_failure_signature = None
        identical_failures = 0

        def submit_next():
            spec = next(source_queue, None)
            if spec is not None:
                futures[executor.submit(process_video, spec)] = spec

        for _ in range(worker_count):
            submit_next()
        completed_count = 0
        while futures:
            future = next(as_completed(futures))
            spec = futures.pop(future)
            completed_count += 1
            try:
                result = future.result()
            except Exception as error:
                signature = f"{type(error).__name__}:{error}"
                identical_failures = identical_failures + 1 if signature == last_failure_signature else 1
                last_failure_signature = signature
                failures.append({"video_id": spec["video_id"], "stage": "inference", "error": repr(error)})
                print(f"[{completed_count}/{len(pending_specs)}] FAILED {spec['video_id']}: {error!r}")
                if identical_failures >= MAX_CONSECUTIVE_IDENTICAL_FAILURES:
                    halt_scheduling = True
                    cancelled = sum(f.cancel() for f in list(futures))
                    futures = {f: s for f, s in futures.items() if not f.cancelled()}
                    print(
                        "Circuit breaker: identical inference error repeated "
                        f"{identical_failures} times; stopped new work and cancelled {cancelled} queued video(s)."
                    )
                if not halt_scheduling:
                    submit_next()
                continue
            last_failure_signature = None
            identical_failures = 0
            if not halt_scheduling:
                submit_next()
            run_results.append(result)
            stats = result["stats"]
            print(
                f"[{completed_count}/{len(pending_specs)}] OK {result['video_id']}: "
                f"{result['frames']} frames, {result['regions']} regions, "
                f"{stats['inference_fps']} FPS, {stats['device']}"
            )
            upload_future = upload_executor.submit(finalize_upload, result)
            upload_futures[upload_future] = result
            while len(upload_futures) >= MAX_PENDING_UPLOADS:
                collect_upload(upload_futures, next(as_completed(upload_futures)))
    while upload_futures:
        collect_upload(upload_futures, next(as_completed(upload_futures)))

print("GPU workers:", len(pipelines), "| Prefetch workers:", worker_count - len(pipelines))
print("Uploaded this run:", len(uploaded_video_ids))
print("Processed this run:", len(run_results), "| Failures:", len(failures))
if run_results:
    total = sum(item["frames"] for item in run_results)
    print("OCR frames:", total, "| overall FPS:", round(total / max(time.perf_counter() - run_started, 1e-6), 3))
if failures:
    print(json.dumps(failures, ensure_ascii=False, indent=2))
    raise RuntimeError("OCR run has inference/upload failures; rerun to resume")

## 5. Audit completion và xem mẫu

Full run chỉ hoàn tất khi mọi thành viên nguồn thực tế có marker hợp lệ và hash
khớp cả hai artifact. Pilot hiển thị một số frame có/không có chữ để kiểm tra box,
dấu tiếng Việt và ngưỡng confidence trước khi bỏ giới hạn video.

In [ ]:
final_files = set() if PREFLIGHT_ONLY else set(retry(
    lambda: api.list_repo_files(OUTPUT_REPO, repo_type="dataset"),
    "refresh OCR files",
))
with ThreadPoolExecutor(max_workers=12) as validator:
    final_flags = list(validator.map(lambda spec: remote_complete(spec, final_files), video_specs))
missing = [spec["video_id"] for spec, okay in zip(video_specs, final_flags) if not okay]
if not PREFLIGHT_ONLY:
    print(f"Remote complete: {len(video_specs) - len(missing)}/{len(video_specs)} videos")
    print("Dataset:", f"https://huggingface.co/datasets/{OUTPUT_REPO}")
    if missing:
        print("Remaining:", len(missing), "sample:", missing[:20])
selected_ids = {spec["video_id"] for spec in selected_specs}
selected_missing = [video_id for video_id in missing if video_id in selected_ids]
if selected_missing and not PREFLIGHT_ONLY:
    raise RuntimeError(f"Selected OCR videos lack valid output: {selected_missing}")
if MAX_VIDEOS_PER_RUN is None and missing and not PREFLIGHT_ONLY:
    raise RuntimeError(f"Missing valid OCR output for {len(missing)} supplied videos")
if PREFLIGHT_ONLY:
    print("Preflight-only run complete; no artifact was uploaded.")
elif MAX_VIDEOS_PER_RUN is not None:
    print("Pilot complete. Review the boxed images below, then set MAX_VIDEOS_PER_RUN=None.")
    from IPython.display import Image as DisplayImage, display
    for result in run_results:
        print(
            "\nQA:", result["video_id"], "| frames:", result["frames"],
            "| with text:", result["frames_with_text"], "| regions:", result["regions"],
        )
        for sample in result["qa_samples"]:
            print(
                sample["frame_uid"], f"t={sample['timestamp_sec']:.2f}s",
                "regions:", sample["region_count"],
                "OCR:", sample["ocr_text"] or NO_TEXT_SENTINEL,
            )
            display(DisplayImage(data=sample["jpeg"]))